# Practical Hierarchical Modelling: End-to-End Analysis

## Learning Objectives

By the end of this notebook you will be able to:

1. Conduct a **complete hierarchical analysis** from exploratory data analysis through model fitting to posterior interpretation.
2. Distinguish **fixed effects**, **random effects**, and **hierarchical Bayes** approaches, and know when each is appropriate.
3. Explain **group-level confounding** and apply the **Mundlak device** to address it.
4. Compare hierarchical models using **WAIC** and **LOO-CV** (connecting back to Module 08).
5. Produce publication-quality visualisations of hierarchical model results: caterpillar plots, shrinkage plots, and posterior predictive checks.

### Prerequisites

- Notebooks 01–03 of this module (partial pooling, varying intercepts, varying slopes).
- Module 08 Notebook 03 (model comparison — WAIC, LOO-CV).

In [ ]:
import sys, os, shutil
from pathlib import Path

os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "sudo apt-get update -qq && sudo apt-get install -y -qq "
        "libcairo2-dev libpango1.0-dev && pip install -q manim pymc arviz ipython==8.21.0"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style

apply_style()

HAS_PYMC = False
try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
    print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")
except Exception as e:
    print(f"PyMC not available: {type(e).__name__}: {e}")


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)

    def apply_manim_config(self):
        from manim import config as mcfg
        mcfg.format = "gif"

    def math_text(self, expr, **kwargs):
        from manim import MathTex, Text
        if self.has_latex:
            return MathTex(expr, **kwargs)
        return Text(expr, **kwargs)

    def save_gifs(self, clean=True):
        local_media = Path("media")
        found = list(local_media.rglob("*.gif")) if local_media.exists() else []
        if not found:
            print("  No new GIFs to save.")
            return
        for gif in found:
            dest = self.gif_dir / gif.name
            shutil.copy2(gif, dest)
            print(f"  \u2713 media/gifs/{gif.name}")
        if clean:
            for sub in ("videos", "images", "Tex"):
                d = local_media / sub
                if d.exists():
                    shutil.rmtree(d, ignore_errors=True)
            print("  Cleaned up local temp render files (kept media/jupyter/).")


cfg = Cfg()
rng = np.random.default_rng(42)

---

## 1. Fixed Effects vs. Random Effects vs. Hierarchical Bayes

Before diving into the analysis, we need to clarify terminology that causes endless confusion in statistics.  The terms "fixed effects" and "random effects" mean different things in different traditions.

### The Econometrics / Frequentist View

| Approach | Model | Key Property |
|----------|-------|--------------|
| **Fixed effects** | Include a dummy variable for each group: $\alpha_j$ is a free parameter. | No pooling.  Consistent even if group effects are correlated with predictors.  Cannot estimate group-level predictors. Uses $J - 1$ degrees of freedom. |
| **Random effects** | Assume $\alpha_j \sim \mathcal{N}(0, \tau^2)$ and estimate $\tau$ by restricted maximum likelihood (REML). | Partial pooling.  More efficient than fixed effects *if* the random-effects assumption holds (no correlation between group effects and predictors). |

### The Bayesian View

In Bayesian statistics, the distinction dissolves: all unknown quantities get distributions.  What frequentists call "random effects" become parameters with a **hierarchical prior**, and what they call "fixed effects" become parameters with **independent, vague priors** (or equivalently, a hierarchical model with $\tau \to \infty$).

The Bayesian hierarchical model subsumes both:
- When the posterior of $\tau$ concentrates near zero → heavy pooling (like complete pooling).
- When $\tau$ is large → little pooling (like fixed effects).
- The data determine how much pooling is appropriate.

### The Hausman Test and Its Bayesian Alternative

In frequentist econometrics, the **Hausman test** decides between fixed and random effects by testing whether the group effects are correlated with the predictors.  If the test rejects, you use fixed effects.

The Bayesian alternative is the **Mundlak device** (Section 4 below), which handles the correlation directly within the hierarchical model — no testing required.

---

## 2. The Dataset: Student Achievement Across Schools

We simulate a dataset loosely inspired by educational research: **students nested in schools**.  Each student has a test score ($y$), and we observe a student-level predictor (hours of study, $x$) and a school-level predictor (school funding per student, $w$).

The data-generating process includes a **group-level confound**: schools with more funding also attract students who study more on average.  This creates a correlation between the group effect and the individual predictor that will bias naive estimates.

In [ ]:
# ── Simulate school data with group-level confounding ──
J = 30           # schools
n_per_school = rng.integers(10, 60, J)  # unbalanced
N = n_per_school.sum()

# School-level: funding (standardised)
funding = rng.normal(0, 1, J)

# True parameters
alpha_bar_true = 65.0    # grand mean score
beta_study_true = 3.0    # effect of study hours
gamma_fund_true = 5.0    # effect of school funding
tau_true = 4.0           # residual between-school SD (after funding)
sigma_true = 8.0         # within-school SD

# School intercepts depend on funding + random component
alpha_true = alpha_bar_true + gamma_fund_true * funding + rng.normal(0, tau_true, J)

# ── CONFOUND: students in well-funded schools study more ──
school_idx = np.concatenate([np.full(n, j, dtype=int) for j, n in enumerate(n_per_school)])
study_hours = 0.5 * funding[school_idx] + rng.normal(0, 1, N)  # correlated!

# Observed scores
mu = alpha_true[school_idx] + beta_study_true * study_hours
score = mu + rng.normal(0, sigma_true, N)

# Package into a DataFrame for convenience
df = pd.DataFrame({
    "score": score,
    "study_hours": study_hours,
    "school": school_idx,
    "funding": funding[school_idx],
})

print(f"Schools: {J}, Students: {N}")
print(f"Correlation(study_hours, funding): {np.corrcoef(study_hours, funding[school_idx])[0,1]:.2f}")
print(f"\nTrue parameters:")
print(f"  beta_study = {beta_study_true}")
print(f"  gamma_fund = {gamma_fund_true}")
print(f"  tau (residual between-school) = {tau_true}")
print(f"  sigma (within-school) = {sigma_true}")

---

## 3. Step 1 — Exploratory Data Analysis

Before fitting any model, we explore the data to understand the group structure, sample sizes, and any patterns that should inform our modelling choices.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) Score distributions by school (subset)
ax = axes[0, 0]
school_means = df.groupby("school")["score"].mean().values
school_ns = df.groupby("school").size().values
order = np.argsort(school_means)
for rank, j in enumerate(order[:15]):  # show first 15 schools
    subset = df[df["school"] == j]["score"]
    ax.scatter(np.full(len(subset), rank), subset, s=8, alpha=0.4, color="steelblue")
    ax.scatter(rank, school_means[j], s=50, color="red", zorder=5)
ax.set_xlabel("School (sorted by mean score)")
ax.set_ylabel("Test score")
ax.set_title("Scores by School (first 15, sorted)")

# (b) Sample sizes per school
ax = axes[0, 1]
ax.bar(range(J), school_ns[order], color="steelblue", alpha=0.7)
ax.set_xlabel("School (sorted by mean score)")
ax.set_ylabel("Number of students")
ax.set_title("Sample Sizes per School")
ax.axhline(school_ns.mean(), color="red", linestyle="--", label=f"Mean = {school_ns.mean():.0f}")
ax.legend()

# (c) Score vs study hours (coloured by school)
ax = axes[1, 0]
for j in range(min(J, 10)):  # colour first 10 schools
    mask = df["school"] == j
    ax.scatter(df.loc[mask, "study_hours"], df.loc[mask, "score"], s=10, alpha=0.4)
ax.set_xlabel("Study hours (standardised)")
ax.set_ylabel("Test score")
ax.set_title("Score vs. Study Hours (10 schools shown)")

# (d) School mean score vs school funding
ax = axes[1, 1]
ax.scatter(funding, school_means, s=school_ns * 2, color="steelblue", alpha=0.7)
ax.set_xlabel("School funding (standardised)")
ax.set_ylabel("Mean test score")
ax.set_title("School Mean Score vs. Funding\n(size ∝ n)")

# Fit trend line
slope, intercept = np.polyfit(funding, school_means, 1)
x_fit = np.linspace(funding.min(), funding.max(), 100)
ax.plot(x_fit, intercept + slope * x_fit, color="red", linewidth=2)

plt.tight_layout()
plt.show()

Key observations from the EDA:

- **School means vary substantially** — there is real between-school variation.
- **Sample sizes are unbalanced** — some schools have 10 students, others 60.  Small schools will benefit most from shrinkage.
- **Score increases with study hours** within schools.
- **School funding predicts school mean scores** — well-funded schools score higher on average.
- **Confound:** students in well-funded schools also study more.  This means the naive within-school slope for study hours may be biased.

---

## 4. The Group-Level Confounding Problem and the Mundlak Device

### The Problem

In our data, study hours are correlated with school funding (a group-level variable).  If we include a varying intercept for school but *don't* account for the school-level mean of study hours, the model will partly attribute the funding effect to study hours, inflating $\hat{\beta}_{\text{study}}$.

Formally, the problem arises when:

$$\text{Cov}(\bar{x}_{\cdot j}, \alpha_j) \neq 0$$

That is, the group mean of the individual predictor is correlated with the group effect.  This violates the "random effects" assumption in frequentist mixed models.

### The Mundlak Device

Mundlak (1978) proposed a simple, elegant solution: **include the group mean of the individual predictor as an additional covariate**:

$$y_i = \alpha_{j[i]} + \beta_W (x_i - \bar{x}_{\cdot j}) + \beta_B \bar{x}_{\cdot j} + \varepsilon_i$$

where:
- $\bar{x}_{\cdot j} = \frac{1}{n_j} \sum_{i: j[i]=j} x_i$ is the group mean of $x$.
- $\beta_W$ is the **within-group** effect of $x$ ("within a school, one more hour of study...").
- $\beta_B$ is the **between-group** effect ("schools whose students study more on average...").

Equivalently, we can write:

$$y_i = \alpha_{j[i]} + \beta_W x_i + (\beta_B - \beta_W) \bar{x}_{\cdot j} + \varepsilon_i$$

### Why This Works

By including $\bar{x}_{\cdot j}$, we **absorb** the group-level variation in $x$ that is confounded with $\alpha_j$.  The coefficient $\beta_W$ now reflects the *pure within-group effect*, free of the group-level confound.

If $\beta_W = \beta_B$, there is no confounding and the Mundlak model reduces to the ordinary varying-intercept model.  If $\beta_W \neq \beta_B$, the naive model would be biased.

---

## 5. Step 2 — Fitting Three Models

We fit three models to compare approaches:

1. **Naive hierarchical:** Varying intercepts, ignoring the confound.
2. **With school-level predictor:** Add funding as a group-level covariate.
3. **Mundlak model:** Add both $x_i$ and $\bar{x}_{\cdot j}$ (the school mean of study hours).

In [ ]:
# Compute school-level mean of study hours (for Mundlak device)
school_mean_study = df.groupby("school")["study_hours"].transform("mean").values

### Model 1: Naive Hierarchical (Varying Intercepts)

In [ ]:
if HAS_PYMC:
    with pm.Model() as model_naive:
        a_bar = pm.Normal("a_bar", mu=65, sigma=10)
        tau = pm.HalfCauchy("tau", beta=5)
        sigma = pm.HalfCauchy("sigma", beta=10)
        beta_study = pm.Normal("beta_study", mu=0, sigma=5)

        z = pm.Normal("z", mu=0, sigma=1, shape=J)
        alpha = pm.Deterministic("alpha", a_bar + z * tau)

        mu_obs = alpha[school_idx] + beta_study * study_hours
        y_obs = pm.Normal("y_obs", mu=mu_obs, sigma=sigma, observed=score)

        trace_naive = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                                target_accept=0.9, return_inferencedata=True)

    print("Model 1 (Naive):")
    print(az.summary(trace_naive, var_names=["a_bar", "tau", "sigma", "beta_study"]))

### Model 2: With School-Level Predictor (Funding)

In [ ]:
if HAS_PYMC:
    with pm.Model() as model_funding:
        a_bar = pm.Normal("a_bar", mu=65, sigma=10)
        gamma_fund = pm.Normal("gamma_fund", mu=0, sigma=5)
        tau = pm.HalfCauchy("tau", beta=5)
        sigma = pm.HalfCauchy("sigma", beta=10)
        beta_study = pm.Normal("beta_study", mu=0, sigma=5)

        z = pm.Normal("z", mu=0, sigma=1, shape=J)
        alpha = pm.Deterministic("alpha", a_bar + gamma_fund * funding + z * tau)

        mu_obs = alpha[school_idx] + beta_study * study_hours
        y_obs = pm.Normal("y_obs", mu=mu_obs, sigma=sigma, observed=score)

        trace_funding = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                                  target_accept=0.9, return_inferencedata=True)

    print("Model 2 (With Funding):")
    print(az.summary(trace_funding, var_names=["a_bar", "gamma_fund", "tau", "sigma", "beta_study"]))

### Model 3: Mundlak Model

In [ ]:
if HAS_PYMC:
    with pm.Model() as model_mundlak:
        a_bar = pm.Normal("a_bar", mu=65, sigma=10)
        gamma_fund = pm.Normal("gamma_fund", mu=0, sigma=5)
        tau = pm.HalfCauchy("tau", beta=5)
        sigma = pm.HalfCauchy("sigma", beta=10)
        beta_study = pm.Normal("beta_study", mu=0, sigma=5)
        beta_xbar = pm.Normal("beta_xbar", mu=0, sigma=5)  # Mundlak term

        # School-level mean of study hours
        xbar_j = np.array([study_hours[school_idx == j].mean() for j in range(J)])

        z = pm.Normal("z", mu=0, sigma=1, shape=J)
        alpha = pm.Deterministic(
            "alpha",
            a_bar + gamma_fund * funding + beta_xbar * xbar_j + z * tau
        )

        mu_obs = alpha[school_idx] + beta_study * study_hours
        y_obs = pm.Normal("y_obs", mu=mu_obs, sigma=sigma, observed=score)

        trace_mundlak = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                                  target_accept=0.9, return_inferencedata=True)

    print("Model 3 (Mundlak):")
    print(az.summary(trace_mundlak, var_names=["a_bar", "gamma_fund", "tau", "sigma", "beta_study", "beta_xbar"]))

---

## 6. Step 3 — Comparing the Three Models

### 6.1 Coefficient Recovery

In [ ]:
if HAS_PYMC:
    fig, ax = plt.subplots(figsize=(10, 5))

    models = {
        "Naive": trace_naive,
        "With Funding": trace_funding,
        "Mundlak": trace_mundlak,
    }
    colours = {"Naive": "red", "With Funding": "steelblue", "Mundlak": "green"}

    y_pos = 0
    yticks = []
    ytick_labels = []

    for name, trace in models.items():
        beta_samples = trace.posterior["beta_study"].values.flatten()
        mean = beta_samples.mean()
        lo, hi = np.percentile(beta_samples, [5.5, 94.5])

        ax.plot([lo, hi], [y_pos, y_pos], color=colours[name], linewidth=3, alpha=0.6)
        ax.scatter(mean, y_pos, color=colours[name], s=100, zorder=5)
        yticks.append(y_pos)
        ytick_labels.append(name)
        y_pos += 1

    ax.axvline(beta_study_true, color="black", linestyle="--", linewidth=2,
               label=f"True $\\beta_{{study}}$ = {beta_study_true}")
    ax.set_yticks(yticks)
    ax.set_yticklabels(ytick_labels)
    ax.set_xlabel(r"$\beta_{\mathrm{study}}$ (effect of study hours)")
    ax.set_title("Recovering the True Study-Hours Effect")
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

The key comparison:

- **Naive model:** The posterior of $\beta_{\text{study}}$ is **biased upward** because it absorbs part of the funding effect (students in well-funded schools study more *and* score higher for other reasons).
- **With Funding:** Including funding as a group-level predictor helps, but doesn't fully resolve the confound because funding explains school means but not the within-school correlation between study hours and school quality.
- **Mundlak model:** Adding $\bar{x}_{\cdot j}$ (school mean of study hours) absorbs the group-level confound.  The posterior of $\beta_{\text{study}}$ should now be centred near the true value.

This demonstrates *why* the Mundlak device matters in practice.

### 6.2 Model Comparison via WAIC

In [ ]:
if HAS_PYMC:
    compare_dict = {
        "Naive": trace_naive,
        "With Funding": trace_funding,
        "Mundlak": trace_mundlak,
    }
    comparison = az.compare(compare_dict, ic="waic")
    print(comparison)
    print()
    az.plot_compare(comparison, figsize=(10, 3))
    plt.title("WAIC Model Comparison")
    plt.tight_layout()
    plt.show()

---

## 7. Step 4 — Visualising the Results

### 7.1 Caterpillar Plot

A **caterpillar plot** ranks the group-level estimates by their posterior mean and displays credible intervals.  It is the standard visualisation for hierarchical model results.

In [ ]:
if HAS_PYMC:
    # Use the Mundlak model (best)
    alpha_post = trace_mundlak.posterior["alpha"].values.reshape(-1, J)
    alpha_mean = alpha_post.mean(axis=0)
    alpha_lo = np.percentile(alpha_post, 5.5, axis=0)
    alpha_hi = np.percentile(alpha_post, 94.5, axis=0)

    # Sort by posterior mean
    order = np.argsort(alpha_mean)

    fig, ax = plt.subplots(figsize=(10, 8))
    for rank, j in enumerate(order):
        colour = "steelblue" if n_per_school[j] >= 25 else "darkorange"
        ax.plot([alpha_lo[j], alpha_hi[j]], [rank, rank], color=colour, linewidth=2, alpha=0.7)
        ax.scatter(alpha_mean[j], rank, color=colour, s=30, zorder=5)
        ax.scatter(alpha_true[j], rank, color="black", s=20, marker="x", zorder=6)

    # Population mean
    abar_post = trace_mundlak.posterior["a_bar"].values.flatten()
    ax.axvline(abar_post.mean(), color="red", linestyle="--", linewidth=1.5,
               label=r"$\bar{\alpha}$ (population mean)")

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color="steelblue", linewidth=2, label=r"$n_j \geq 25$"),
        Line2D([0], [0], color="darkorange", linewidth=2, label=r"$n_j < 25$"),
        Line2D([0], [0], color="black", marker="x", linestyle="None", label="True $\\alpha_j$"),
        Line2D([0], [0], color="red", linestyle="--", label=r"$\bar{\alpha}$"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

    ax.set_xlabel(r"School intercept $\alpha_j$")
    ax.set_ylabel("School (ranked by posterior mean)")
    ax.set_title("Caterpillar Plot: School-Level Intercepts (Mundlak Model)")
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()

Notice that:
- Small schools (orange) have **wider credible intervals** — more uncertainty.
- Small schools are also **shrunk more** toward the population mean (red dashed line).
- The true values (black crosses) generally fall within the 89% credible intervals.

### 7.2 Shrinkage Plot

In [ ]:
if HAS_PYMC:
    # No-pooling estimates (school raw means after removing study effect)
    beta_est = trace_mundlak.posterior["beta_study"].values.flatten().mean()
    residuals = score - beta_est * study_hours
    alpha_nopool = np.array([residuals[school_idx == j].mean() for j in range(J)])

    fig, ax = plt.subplots(figsize=(8, 8))

    # 45-degree line (no shrinkage)
    lims = [min(alpha_nopool.min(), alpha_mean.min()) - 2,
            max(alpha_nopool.max(), alpha_mean.max()) + 2]
    ax.plot(lims, lims, color="grey", linestyle=":", linewidth=1)

    # Population mean lines
    ax.axhline(abar_post.mean(), color="gold", linestyle="--", linewidth=1, alpha=0.5)
    ax.axvline(abar_post.mean(), color="gold", linestyle="--", linewidth=1, alpha=0.5)

    # Points: size proportional to n_j
    ax.scatter(alpha_nopool, alpha_mean, s=n_per_school * 3, color="steelblue",
               alpha=0.7, edgecolors="white", linewidth=0.5)

    ax.set_xlabel("No-pooling estimate (raw school mean)")
    ax.set_ylabel("Partial-pooling estimate (posterior mean)")
    ax.set_title("Shrinkage Plot\n(size ∝ school sample size)")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

Points that deviate from the 45-degree line (grey dotted) have been shrunk.  Small schools (small dots) are pulled further from the diagonal toward the population mean (gold dashed lines).  Large schools sit close to the diagonal — their data is strong enough to resist the pull.

### 7.3 Posterior Predictive Check

In [ ]:
if HAS_PYMC:
    with model_mundlak:
        ppc = pm.sample_posterior_predictive(trace_mundlak, random_seed=42)

    fig, ax = plt.subplots(figsize=(9, 4))
    az.plot_ppc(az.from_pymc3(posterior_predictive=ppc, model=model_mundlak) if hasattr(az, 'from_pymc3') else ppc,
                ax=ax, num_pp_samples=100, alpha=0.05)
    ax.set_title("Posterior Predictive Check (Mundlak Model)")
    ax.set_xlabel("Test score")
    plt.tight_layout()
    plt.show()

---

## 8. When to Use Hierarchical Models: Practical Guidelines

### Use Hierarchical Models When:

- Data has **natural group structure** (students in schools, patients in hospitals, items in categories).
- **Sample sizes vary** across groups — small groups benefit from borrowing strength.
- You want to **predict for new groups** (e.g., a new school joins the study).
- You want to **estimate group-level variation** ($\tau$), not just control for it.
- Group effects may be **correlated with slopes** (use Mundlak or varying slopes).

### Don't Use Hierarchical Models When:

- There are **very few groups** ($J < 5$): not enough groups to estimate $\tau$ reliably. Use fixed effects or complete pooling.
- Groups are **not exchangeable** even after conditioning on covariates.
- The computational cost is unjustified (e.g., $J = 2$ groups with large $n$).

### Practical Workflow Summary

1. **EDA:** Examine group sizes, within/between variation, potential confounds.
2. **Decide on varying effects:** Start with varying intercepts.  Add varying slopes if there is substantive reason to expect heterogeneous effects.
3. **Address confounding:** If individual-level predictors are correlated with group identity, use the Mundlak device or include group-level predictors.
4. **Choose parameterisation:** Non-centered by default, especially with many groups or few data per group.
5. **Fit and diagnose:** Check ESS, R-hat, divergences, trace plots.
6. **Compare models:** WAIC or LOO-CV for out-of-sample predictive performance.
7. **Interpret and visualise:** Caterpillar plots, shrinkage plots, posterior predictive checks.

---

## 9. Key Takeaways

1. **Fixed effects, random effects, and hierarchical Bayes** are three approaches to the same problem. The Bayesian framework subsumes the other two by letting $\tau$ be learned from data.

2. **Group-level confounding** occurs when individual-level predictors are correlated with group effects. The **Mundlak device** (including group means of individual predictors) is a simple, effective solution.

3. A complete hierarchical analysis follows: EDA → model specification → fitting → diagnostics → comparison → interpretation.

4. **Caterpillar plots** and **shrinkage plots** are the standard visualisations for hierarchical model results. They directly show the partial-pooling behaviour.

5. Use **WAIC or LOO-CV** to compare hierarchical models with different levels of complexity.

## Further Reading

- Mundlak, Y. (1978). *On the pooling of time series and cross section data*. Econometrica.
- Bell, A. & Jones, K. (2015). *Explaining fixed effects: Random effects modelling of time-series cross-sectional and panel data*. Political Science Research and Methods.
- McElreath, R. (2020). *Statistical Rethinking*, Ch. 14 (bonus: Mundlak machine).
- Gelman, A. & Hill, J. (2007). *Data Analysis Using Regression and Multilevel/Hierarchical Models*.

**Next module:** We move to **causal inference** (Module 10), where hierarchical models reappear as tools for estimating causal effects while adjusting for group structure.